# 03 - Whale Alert: Large Trade Detection

**PolyWatch — Market Integrity Monitoring System**  
**Member C — Core Algorithm Module**

This notebook demonstrates the Whale Alert module for detecting
suspicious large-volume trading activity.

## Detection Methods
1. Single large trade detection
2. Cumulative volume spikes within time windows
3. Directional bias (concentrated buying/selling)
4. Price impact analysis around whale trades

**Note**: Since the data pipeline currently only collects price data,
we use `simulate_trades_from_prices()` to generate synthetic trades
based on price movements.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from core_analysis.whale_alert import (
    WhaleAlert, WhaleConfig,
    simulate_trades_from_prices, run_whale_analysis,
)
from core_analysis.db_interface import get_price_series

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Imports OK')

## 1. Load Price Data & Generate Synthetic Trades

In [ ]:
slug = 'presidential-election-winner-2024'
price_df = get_price_series(slug)
price_series = price_df['price']

# Generate synthetic trades from price movements
trades = simulate_trades_from_prices(price_series)

print(f'Market: {slug}')
print(f'Price data points: {len(price_series)}')
print(f'Simulated trades: {len(trades)}')
print(f'\nTrade size stats:')
print(trades['trade_size'].describe().round(2))
print(f'\nSide distribution:')
print(trades['side'].value_counts())

## 2. Run Whale Detection

In [ ]:
result = run_whale_analysis(trades, price_series)

print('=== Whale Detection Summary ===')
summary = result['summary']
for k, v in summary.items():
    if k != 'config':
        print(f'  {k}: {v}')

print(f'\n=== Config ===')
for k, v in summary['config'].items():
    print(f'  {k}: {v}')

## 3. Visualize Whale Trades

In [ ]:
whale_trades = result['whale_trades']
flagged = result['trades_flagged']

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Panel 1: Price with whale trade markers
ax = axes[0]
ax.plot(price_series.index, price_series.values, linewidth=0.7, color='steelblue', label='Price')
if not whale_trades.empty:
    medium = whale_trades[whale_trades['whale_severity'] == 'medium']
    high = whale_trades[whale_trades['whale_severity'] == 'high']
    if not medium.empty:
        ax.scatter(medium['timestamp'], medium['price'],
                   color='orange', s=30, zorder=5, label=f'Whale (medium): {len(medium)}')
    if not high.empty:
        ax.scatter(high['timestamp'], high['price'],
                   color='red', s=50, zorder=5, marker='^', label=f'Whale (high): {len(high)}')
ax.set_title('Price with Whale Trades', fontsize=13)
ax.set_ylabel('Price')
ax.legend(loc='upper left')

# Panel 2: Trade volume
ax = axes[1]
normal_trades = flagged[~flagged['is_whale']]
ax.bar(normal_trades['timestamp'], normal_trades['trade_size'],
       width=0.03, color='steelblue', alpha=0.3, label='Normal')
if not whale_trades.empty:
    ax.bar(whale_trades['timestamp'], whale_trades['trade_size'],
           width=0.03, color='red', alpha=0.7, label='Whale')
ax.axhline(y=500, color='red', linestyle='--', linewidth=0.8, label='Threshold (500)')
ax.set_title('Trade Sizes', fontsize=13)
ax.set_ylabel('Trade Size')
ax.set_xlabel('Date')
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 4. Trade Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all trade sizes
ax = axes[0]
ax.hist(trades['trade_size'], bins=50, color='steelblue', alpha=0.7, edgecolor='navy')
ax.axvline(x=500, color='red', linestyle='--', label='Whale Threshold')
ax.set_xlabel('Trade Size')
ax.set_ylabel('Count')
ax.set_title('Trade Size Distribution')
ax.legend()

# Log-scale
ax = axes[1]
ax.hist(trades['trade_size'], bins=50, color='steelblue', alpha=0.7, edgecolor='navy', log=True)
ax.axvline(x=500, color='red', linestyle='--', label='Whale Threshold')
ax.set_xlabel('Trade Size')
ax.set_ylabel('Count (log scale)')
ax.set_title('Trade Size Distribution (Log Scale)')
ax.legend()

plt.tight_layout()
plt.show()

## 5. Directional Bias Events

In [ ]:
directional_events = result['directional_events']
print(f'Directional bias events: {len(directional_events)}')

if directional_events:
    bias_df = pd.DataFrame(directional_events)
    print(bias_df[['time_start', 'bias_side', 'buy_pct', 'sell_pct',
                    'total_volume', 'severity']].head(20).to_string())
else:
    print('No directional bias detected.')

## 6. Price Impact of Whale Trades

In [ ]:
impact_df = result['price_impact_df']

if not impact_df.empty:
    print('Price impact of whale trades:')
    cols = ['timestamp', 'trade_size', 'whale_severity', 'price_before', 'price_after', 'price_impact']
    available_cols = [c for c in cols if c in impact_df.columns]
    print(impact_df[available_cols].head(20).to_string())
    
    # Plot
    if 'price_impact' in impact_df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.scatter(impact_df['trade_size'], impact_df['price_impact'],
                   alpha=0.6, color='coral', edgecolor='darkred')
        ax.set_xlabel('Trade Size')
        ax.set_ylabel('Price Impact (absolute)')
        ax.set_title('Trade Size vs Price Impact')
        plt.tight_layout()
        plt.show()
else:
    print('No whale trades to analyze for price impact.')

---

## Summary

The Whale Alert module successfully detects:
- Individual large trades exceeding the size threshold
- Cumulative volume spikes within rolling time windows
- Directional bias (coordinated buying/selling)
- Price impact of whale trades

**Limitation**: Currently using simulated trades generated from price
movements. Results will be more meaningful once real trade data from
the Polymarket CLOB API is integrated.